# CIFAR-10: Augmentation & Optimization Experiments

**Paths chosen:** Data Augmentation (CutMix/Cutout/MixUp) and Optimization (SGD vs Adam, LR Schedules)

This Colab notebook is written to run end-to-end on Colab GPU. Run cells in order.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torchvision
from torchvision import transforms, datasets, models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import itertools
import time
import copy
import os

print('Torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
def imshow(img, title=None):
    img = img / 2 + 0.5  # unnormalize if using mean/std 0.5
    npimg = img.numpy()
    plt.figure(figsize=(4,4))
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    if title: plt.title(title)
    plt.axis('off')

from matplotlib import rcParams
rcParams['figure.dpi'] = 100

def plot_curves(history, metric='train_loss'):
    plt.figure(figsize=(6,4))
    for k,v in history.items():
        plt.plot(v, label=k)
    plt.legend()
    plt.title(metric)
    plt.show()

In [ ]:
########################################
# Data transforms and augmentation setup
########################################

BATCH_SIZE = 128
NUM_WORKERS = 2

# Basic transforms (baseline)
train_transform_basic = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])

# Eval transform
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])

# Advanced: Cutout function (simple implementation)
import random
import torch

def cutout(img, n_holes=1, length=8):
    # img: tensor CxHxW
    h = img.size(1)
    w = img.size(2)
    mask = torch.ones((h, w), dtype=torch.float32)
    for _ in range(n_holes):
        y = random.randint(0, h - 1)
        x = random.randint(0, w - 1)
        y1 = max(0, y - length // 2)
        y2 = min(h, y1 + length)
        x1 = max(0, x - length // 2)
        x2 = min(w, x1 + length)
        mask[y1:y2, x1:x2] = 0.
    mask = mask.expand_as(img)
    img = img * mask
    return img

# We'll implement CutMix within the training loop (function provided in notebook)

def rand_bbox(size, lam):
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = np.int(W * cut_rat)
    cut_h = np.int(H * cut_rat)

    # uniform
    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2

In [ ]:
########################################
# Data loading (CIFAR-10)
########################################
from torchvision.datasets import CIFAR10

def get_dataloaders(transform_train):
    trainset = CIFAR10(root='./data', train=True, download=True, transform=transform_train)
    testset = CIFAR10(root='./data', train=False, download=True, transform=test_transform)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    testloader = torch.utils.data.DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    return trainloader, testloader

# Example: create dataloaders with basic transforms
trainloader, testloader = get_dataloaders(train_transform_basic)

# show a batch
dataiter = iter(trainloader)
images, labels = next(dataiter)
imshow(torchvision.utils.make_grid(images[:16]))
print('Labels:', labels[:16].numpy())

In [ ]:
########################################
# Model: ResNet18 (preact adaptation for CIFAR10)
########################################

def get_model(pretrained=False, num_classes=10):
    model = models.resnet18(pretrained=pretrained)
    # adjust first conv for CIFAR10 (3x32x32) - change kernel_size=3, stride=1, padding=1
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = get_model().to(device)
print(model)

# Count params
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Trainable parameters:', count_parameters(model))

In [ ]:
########################################
# Training & evaluation utilities
########################################
import torch.nn.functional as F

def train_one_epoch(model, loader, optimizer, criterion, device, epoch, use_cutmix=False, cutmix_prob=0.5, alpha=1.0):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for i, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        if use_cutmix and np.random.rand() < cutmix_prob:
            lam = np.random.beta(alpha, alpha)
            rand_index = torch.randperm(inputs.size()[0]).to(device)
            target_a = targets
            target_b = targets[rand_index]
            bbx1, bby1, bbx2, bby2 = rand_bbox(inputs.size(), lam)
            inputs[:, :, bby1:bby2, bbx1:bbx2] = inputs[rand_index, :, bby1:bby2, bbx1:bbx2]
            # adjust lambda to exactly match pixel ratio
            lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (inputs.size()[-1] * inputs.size()[-2]))
            outputs = model(inputs)
            loss = criterion(outputs, target_a) * lam + criterion(outputs, target_b) * (1. - lam)
        else:
            outputs = model(inputs)
            loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        # For CutMix case, accuracy computed w.r.t. target_a only (approx)
        correct += predicted.eq(targets).sum().item()
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            all_preds.append(predicted.cpu().numpy())
            all_labels.append(targets.cpu().numpy())
    acc = correct / total
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    return acc, all_labels, all_preds

In [ ]:
########################################
# Experiment runner (two experiments required)
########################################

def run_experiment(augment='basic', optimizer_name='SGD', epochs=10, lr=0.1, weight_decay=5e-4, use_cutmix=False):
    # choose transform
    if augment == 'basic':
        transform_train = train_transform_basic
    elif augment == 'cutout':
        # we embed cutout inside a custom dataset transform by applying post-Tensor transform
        def transform_with_cutout(x):
            x = train_transform_basic(x)
            x = cutout(x, n_holes=1, length=16)
            return x
        transform_train = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                              transforms.RandomHorizontalFlip(),
                                              transforms.ToTensor(),
                                              transforms.Lambda(lambda x: cutout(x, n_holes=1, length=16)),
                                              transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))])
    else:
        transform_train = train_transform_basic

    trainloader, testloader = get_dataloaders(transform_train)
    model = get_model().to(device)
    criterion = nn.CrossEntropyLoss()
    if optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    elif optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)

    scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {'train_loss': [], 'train_acc': [], 'val_acc': []}
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    for epoch in range(epochs):
        loss, acc = train_one_epoch(model, trainloader, optimizer, criterion, device, epoch, use_cutmix=use_cutmix)
        val_acc, _, _ = evaluate(model, testloader, device)
        history['train_loss'].append(loss)
        history['train_acc'].append(acc)
        history['val_acc'].append(val_acc)
        scheduler.step()
        print(f'Epoch {epoch+1}/{epochs}: loss={loss:.4f}, train_acc={acc:.4f}, val_acc={val_acc:.4f}')
        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_model_wts)
    print('Best val acc:', best_acc)
    return model, history, trainloader, testloader

# Example quick run for debugging (use small epochs)
# model_res, hist, tr, te = run_experiment(augment='basic', optimizer_name='SGD', epochs=2, lr=0.01, use_cutmix=False)


In [ ]:
########################################
# Evaluation utilities: confusion matrix and per-class accuracy
########################################

import seaborn as sns

classes = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck')

def plot_confusion_matrix(cm, classes):
    plt.figure(figsize=(7,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show()

# compute per-class acc

def per_class_accuracy(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    per_class = cm.diagonal() / cm.sum(axis=1)
    return per_class, cm

# Example: after a trained model, run evaluation and plot
# acc, y_true, y_pred = evaluate(model_res, te, device)
# per_acc, cm = per_class_accuracy(y_true, y_pred)
# print('Overall acc', acc)
# print('Per-class acc', per_acc)
# plot_confusion_matrix(cm, classes)

In [ ]:
########################################
# Grad-CAM (simple) for explainability
########################################

def generate_gradcam(model, input_tensor, target_class=None, layer_name='layer4'):
    model.eval()
    features = None
    gradients = None

    def forward_hook(module, inp, outp):
        nonlocal features
        features = outp.detach()
    def backward_hook(module, grad_in, grad_out):
        nonlocal gradients
        gradients = grad_out[0].detach()

    # register hooks
    target_layer = dict(model.named_modules())[layer_name]
    fh = target_layer.register_forward_hook(forward_hook)
    bh = target_layer.register_backward_hook(backward_hook)

    outputs = model(input_tensor)
    if target_class is None:
        target_class = outputs.argmax().item()
    loss = outputs[0, target_class]
    model.zero_grad()
    loss.backward()

    pooled_grads = torch.mean(gradients, dim=[0,2,3])
    for i in range(features.shape[1]):
        features[0,i,:,:] *= pooled_grads[i]
    heatmap = features[0].mean(dim=0).cpu().numpy()
    heatmap = np.maximum(heatmap, 0)
    heatmap /= np.max(heatmap) + 1e-9

    fh.remove(); bh.remove()
    return heatmap

# Example usage (after training):
# img, label = next(iter(te))
# inp = img[0:1].to(device)
# hm = generate_gradcam(model_res, inp)
# plt.imshow(hm, cmap='jet')

In [ ]:
########################################
# Save and load model utility (for Colab)
########################################

def save_model(model, path='best_model.pth'):
    torch.save(model.state_dict(), path)

def load_model(path='best_model.pth'):
    model = get_model().to(device)
    model.load_state_dict(torch.load(path, map_location=device))
    return model

print('Notebook built. To run experiments, call run_experiment(augment=..., optimizer_name=..., epochs=...)')